# Mission 16 - 기본 미션: ONNX 추론 검증

**목표**: `onnxruntime`으로 MNIST test set 추론 → 정확도 95%+ 확인

In [1]:
# !pip install onnxruntime numpy torchvision

In [2]:
import numpy as np
import onnxruntime as ort
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

print(f"onnxruntime version: {ort.__version__}")
print(f"Available providers: {ort.get_available_providers()}")

onnxruntime version: 1.24.3
Available providers: ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


## 1. ONNX 세션 로드

In [3]:
ONNX_PATH = './data/models/mission_16_mnist_cnn.onnx'

session = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape
output_name = session.get_outputs()[0].name
output_shape = session.get_outputs()[0].shape

print(f"Input  : {input_name} {input_shape}")
print(f"Output : {output_name} {output_shape}")

Input  : input ['batch_size', 1, 28, 28]
Output : output ['batch_size', 10]


## 2. MNIST test set 로드 (정규화 포함)

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = torchvision.datasets.MNIST(
    root='./data/mnist_data', train=False, download=True, transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
print(f"Test set: {len(test_dataset):,}장")

Test set: 10,000장


## 3. 전체 test set 추론 → 정확도

In [5]:
correct = 0
total = 0

for images, labels in test_loader:
    # numpy 변환 (NCHW float32)
    x = images.numpy().astype(np.float32)
    
    # ONNX 추론
    outputs = session.run([output_name], {input_name: x})[0]  # (N, 10)
    preds = np.argmax(outputs, axis=1)
    
    correct += (preds == labels.numpy()).sum()
    total += len(labels)

accuracy = correct / total
print("=" * 40)
print(f"ONNX 모델 추론 결과")
print(f"총 {total:,}장 중 {correct:,}장 정답")
print(f"정확도: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("=" * 40)

assert accuracy >= 0.95, f"정확도 {accuracy:.4f} < 95% 기준 미달"
print("✓ 95% 이상 달성")

ONNX 모델 추론 결과
총 10,000장 중 9,909장 정답
정확도: 0.9909 (99.09%)
✓ 95% 이상 달성


## 4. 샘플 추론 (시각화)

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 첫 배치에서 16개 샘플
images_sample, labels_sample = next(iter(test_loader))
x_sample = images_sample[:16].numpy().astype(np.float32)
outputs_sample = session.run([output_name], {input_name: x_sample})[0]
preds_sample = np.argmax(outputs_sample, axis=1)

fig = make_subplots(rows=2, cols=8, horizontal_spacing=0.02, vertical_spacing=0.08)

for i in range(16):
    row, col = divmod(i, 8)
    img = images_sample[i, 0].numpy()
    # [-1, 1] 범위 → [0, 255] 정규화 (시각화용)
    img_display = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)

    is_correct = preds_sample[i] == labels_sample[i].item()
    border_color = 'green' if is_correct else 'red'

    fig.add_trace(
        go.Heatmap(
            z=img_display[::-1],  # y축 반전 (이미지 좌표계)
            colorscale='Gray',
            showscale=False,
        ),
        row=row + 1, col=col + 1
    )
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, row=row + 1, col=col + 1)

    # 예측/정답 레이블 (subplot 제목 대신 annotation 사용)
    fig.add_annotation(
        text=f"<b style='color:{border_color}'>pred:{preds_sample[i]} true:{labels_sample[i].item()}</b>",
        xref=f"x{i+1}", yref=f"y{i+1}",
        x=14, y=-3,
        showarrow=False,
        font=dict(size=9),
        xanchor='center',
    )

fig.update_layout(
    title_text='ONNX Inference Results (green=correct, red=wrong)',
    height=320,
    width=1100,
    margin=dict(t=50, b=10, l=10, r=10),
    paper_bgcolor='white',
    plot_bgcolor='white',
)

fig.write_image('./screenshots/onnx_inference_sample.png', scale=2)
fig.show()
print("스크린샷 저장: ./screenshots/onnx_inference_sample.png")

스크린샷 저장: ./screenshots/onnx_inference_sample.png
